In [1]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

# Initialize geocoder
geolocator = Nominatim(user_agent="hospital_locator")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)  # Add delay to respect usage policy

def get_coordinates(address):
    try:
        location = geocode(address)
        if location:
            return (location.latitude, location.longitude)
        else:
            return (None, None)
    except Exception as e:
        print(f"Error geocoding {address}: {e}")
        return (None, None)

# Load hospitals from hospitals.json
import json
with open("hospitals_raw.json", "r") as file:
    hospitals = json.load(file)

# Add coordinates to each hospital
for hospital in hospitals:
    address = hospital["Adresse"]
    lat, lon = get_coordinates(address)
    
    # If first attempt fails, try with just city and postal code
    if lat is None or lon is None:
        city_part = address.split(",")[-1].strip()
        lat, lon = get_coordinates(city_part)
    
    hospital["Coordinates"] = (lat, lon)
    print(f"Processed: {hospital['Name']} - Coordinates: {hospital['Coordinates']}")
    time.sleep(1)  # Be nice to the geocoding service

# Print results
print("\nResults:")
for hospital in hospitals:
    print(f"{hospital['Name']}: {hospital['Coordinates']}")

Processed: Krankenhaus Salem Evang. Stadtmission Heidelberg - Coordinates: (49.4231957, 8.6846485)
Processed: St. Elisabeth Klinik - Coordinates: (49.4225321, 8.6819217)
Processed: Klinikum der Universität Heidelberg - Coordinates: (49.4199193, 8.6676297)
Processed: UKHD: Zentrum für Psychosoziale Medizin - Coordinates: (49.409652, 8.6856575)
Processed: Nierenzentrum Heidelberg - Coordinates: (49.4141046, 8.6658128)
Processed: St. Josefskrankenhaus - Coordinates: (49.4025657, 8.688835)
Processed: Kurpfalzkrankenhaus Heidelberg gGmbH - Coordinates: (49.4131515, 8.6516972)
Processed: Kliniken Schmieder Heidelberg GmbH & Co KG - Coordinates: (49.3943579, 8.7085351)
Processed: Agaplesion Bethanien Krankenhaus Heidelberg - Coordinates: (49.39163, 8.6910085)
Processed: Thoraxklinik Rohrbach - Coordinates: (49.3787569, 8.6916291)
Processed: UKHD: Zentrum für Orthopädie, Unfallchirurgie und Paraplegiologie - Coordinates: (49.4086936, 8.7736009)
Processed: GRN - Klinik Schwetzingen - Coordinate

RateLimiter caught an error, retrying (0/2 tries). Called with (*('Knittlinger Steige 21, 75433 Maulbronn',), **{}).
Traceback (most recent call last):
  File "/Users/felixkohlhas/Documents/Projects/Random/MIT_Hackathon_2025/.venv/lib/python3.9/site-packages/geopy/adapters.py", line 298, in get_text
    page = self.urlopen(req, timeout=timeout)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/urllib/request.py", line 517, in open
    response = self._open(req, data)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/urllib/request.py", line 534, in _open
    result = self._call_chain(self.handle_open, protocol, protocol +
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/urllib/request.py", line 494, in _call_chain
    result = func(*args)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.frame

Processed: Kinderzentrum Maulbronn - Coordinates: (49.0005873, 8.8011059)
Processed: Tagesklinik Landau - Coordinates: (49.2008497, 8.1120717)
Processed: Klinikum Landau-Südliche Weinstraße - Klinik Landau - Coordinates: (49.2058562, 8.1038746)
Processed: Vinzentius-Krankenhaus - Coordinates: (49.1903125, 8.1142225)
Processed: Evangelisches Krankenhaus Elisabethenstift gGmbH - Coordinates: (49.8739094, 8.6651328)
Processed: Klinikum Darmstadt - Coordinates: (49.8746212, 8.6468914)
Processed: Klinik Bad Gleisweiler - Coordinates: (49.2420344, 8.0608101)
Processed: Klinik am Rathenauplatz - Coordinates: (49.1316554, 9.2189336)

Results:
Krankenhaus Salem Evang. Stadtmission Heidelberg: (49.4231957, 8.6846485)
St. Elisabeth Klinik: (49.4225321, 8.6819217)
Klinikum der Universität Heidelberg: (49.4199193, 8.6676297)
UKHD: Zentrum für Psychosoziale Medizin: (49.409652, 8.6856575)
Nierenzentrum Heidelberg: (49.4141046, 8.6658128)
St. Josefskrankenhaus: (49.4025657, 8.688835)
Kurpfalzkrankenh

In [3]:
! pip install geopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [geopy]


In [2]:
# Write hospitals to json file
import json

with open('hospitals.json', 'w') as f:
    json.dump(hospitals, f, indent=4)
    print("Hospitals data written to hospitals.json")

Hospitals data written to hospitals.json


In [7]:
import json
import random
from datetime import datetime, timedelta
import time

def convert_hospital_data(input_file, output_file):
    # Load the input JSON file
    with open(input_file, 'r', encoding='utf-8') as f:
        hospitals = json.load(f)
    
    # Get current time for reference
    now = datetime.utcnow()
    
    # Process each hospital entry
    converted_hospitals = []
    for hospital in hospitals:
        # Generate random last_update time between now and -6000 seconds
        random_seconds = random.randint(0, 6000)
        last_update = now - timedelta(seconds=random_seconds)
        last_update_str = last_update.strftime("%Y-%m-%dT%H:%M:%SZ")
        
        try:
            beds = int(hospital.get("Bettenanzahl", 0))
            used_beds = random.randint(0, beds)
        except:
            used_beds = 0

        # Create the converted hospital entry
        converted_hospital = {
            "name": hospital.get("Name", ""),
            "address": hospital.get("Adresse", ""),
            "total_beds": hospital.get("Bettenanzahl", 0),
            "used_beds": used_beds,
            "last_update": last_update_str,
            "last_update_seconds": random_seconds,
            "status": "available",
            "coordinates": hospital.get("Coordinates", [])
        }
        
        converted_hospitals.append(converted_hospital)
    
    # Save the converted data to the output file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(converted_hospitals, f, indent=4, ensure_ascii=False)

# Example usage
input_json_file = "hospitals.json"
output_json_file = "hospitals_v2.json"
convert_hospital_data(input_json_file, output_json_file)